# MyoLab-AI — GRABMyo leakage-safe classical baseline
## Standalone notebook for Primary-4 × Forearm-16 × Feature Set 14

Notebook này chạy trong một notebook/Colab mới. Nó không phụ thuộc vào state của ETL notebook.

## Input bắt buộc

```text
/content/drive/MyDrive/MyoLab-AI-data/
└── grabmyo-physionet-v1.1.0/
    ├── outputs/
    │   └── grabmyo-primary4-forearm16-fall14.npz
    └── manifests/
        ├── grabmyo-record-coverage.csv
        ├── grabmyo-feature-exclusion-report.csv
        └── grabmyo-etl-final-gate.json
```

## Modeling policy

1. Không random split theo window hoặc trial.
2. Frozen split là subject-level `34 train / 9 validation`; cả ba session của một subject ở cùng partition.
3. Grouped CV chỉ chạy trong 34 training subjects.
4. Scaler nằm trong `sklearn.pipeline.Pipeline` và chỉ fit trên training fold.
5. Model/feature arm được chọn bằng **trial-level macro-F1 từ OOF grouped CV**.
6. Frozen validation chỉ mở một lần sau selection.
7. Báo cáo cả window, trial, subject và session metrics.
8. Không test set, không tuning trên validation, không pooled training, không clinical use.

## Core feature arms

- `F-TD8`: `16 × 8 = 128` dimensions.
- `F-SP6`: `16 × 6 = 96` dimensions.
- `F-ALL14`: `16 × 14 = 224` dimensions.
- `F-NO-MOMENTS12`: `16 × 12 = 192` dimensions.

## Core models

- Dummy most frequent.
- LDA shrinkage.
- Logistic regression.
- Linear SVM.
- Random forest là optional vì full dataset lớn.

> Đây là engineering/research baseline, không phải clinical model.


## CELL 1 — Environment setup

**Input:** Python/Colab runtime.  
**Output:** modeling dependencies.


In [1]:
# === CELL 1: Environment setup ===
from __future__ import annotations

import importlib.util
import subprocess
import sys

_REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "joblib": "joblib",
}
_missing = [
    pip_name for import_name, pip_name in _REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(import_name) is None
]
if _missing:
    print("[INFO] Installing missing packages:", _missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])
print("[PASS] Required packages are available.")


[PASS] Required packages are available.


## CELL 2 — Imports, Drive mount, configuration và governance

**Input:** ETL artifacts trên Drive hoặc path override.  
**Output:** standalone run directories và frozen modeling authorization.


In [2]:
# === CELL 2: Imports, paths, configuration and governance ===
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import shutil
import time
import warnings
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

from sklearn.base import clone
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive, files  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

# Governance
MODEL_FITTING_AUTHORIZED = True
OPEN_FROZEN_VALIDATION_AFTER_SELECTION = True
HYPERPARAMETER_TUNING_ALLOWED = False
TEST_SET_OPENED = False
POOLED_TRAINING_ALLOWED = False
CLINICAL_USE_ALLOWED = False

assert MODEL_FITTING_AUTHORIZED is True
assert OPEN_FROZEN_VALIDATION_AFTER_SELECTION is True
assert HYPERPARAMETER_TUNING_ALLOWED is False
assert TEST_SET_OPENED is False
assert POOLED_TRAINING_ALLOWED is False
assert CLINICAL_USE_ALLOWED is False

# Run configuration
RUN_ID = "grabmyo-primary4-baseline-v1"
RANDOM_SEED = 4302
N_SPLITS = 5
N_JOBS = -1
RUN_RANDOM_FOREST = False
FORCE_RECOMPUTE_CV = False
DOWNLOAD_HANDOFF_TO_BROWSER = False
QUICK_SMOKE_TEST = os.environ.get("GRABMYO_BASELINE_SMOKE", "0") == "1"

DATASET_ID = "grabmyo-physionet-v1.1.0"
EXPECTED_DATASET_VIEW_ID = "grabmyo-primary4-forearm16-fall14-v1"
EXPECTED_CLASS_ORDER = (
    "rest", "hand_close", "wrist_flexion", "wrist_extension"
)
EXPECTED_SUBJECT_COUNT = 43
EXPECTED_TRAIN_SUBJECTS = 34
EXPECTED_VALIDATION_SUBJECTS = 9
EXPECTED_FEATURE_DIM = 224
EXPECTED_WINDOWS_PER_RECORD = 48

DRIVE_DATASET_ROOT = (
    Path("/content/drive/MyDrive/MyoLab-AI-data") / DATASET_ID
    if Path("/content/drive/MyDrive").exists()
    else Path.cwd() / DATASET_ID
)

NPZ_PATH_OVERRIDE = os.environ.get("GRABMYO_NPZ_PATH", "").strip()
DEFAULT_NPZ_PATH = DRIVE_DATASET_ROOT / "outputs" / "grabmyo-primary4-forearm16-fall14.npz"
NPZ_PATH = Path(NPZ_PATH_OVERRIDE) if NPZ_PATH_OVERRIDE else DEFAULT_NPZ_PATH

RECORD_COVERAGE_PATH = DRIVE_DATASET_ROOT / "manifests" / "grabmyo-record-coverage.csv"
EXCLUSION_REPORT_PATH = DRIVE_DATASET_ROOT / "manifests" / "grabmyo-feature-exclusion-report.csv"
ETL_FINAL_GATE_PATH = DRIVE_DATASET_ROOT / "manifests" / "grabmyo-etl-final-gate.json"

if IN_COLAB:
    RUNTIME_ROOT = Path("/content/data") / DATASET_ID / "baseline" / RUN_ID
else:
    RUNTIME_ROOT = Path.cwd() / ".grabmyo-baseline-runtime" / RUN_ID

PERSIST_ROOT = DRIVE_DATASET_ROOT / "outputs" / "baseline" / RUN_ID
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)

if QUICK_SMOKE_TEST:
    N_SPLITS = 3

np.random.seed(RANDOM_SEED)

print("=" * 92)
print("[CELL 2] GRABMyo BASELINE CONFIGURATION")
print("=" * 92)
print(f"NPZ path                    : {NPZ_PATH}")
print(f"Runtime output              : {RUNTIME_ROOT}")
print(f"Persistent output           : {PERSIST_ROOT}")
print(f"Grouped CV folds            : {N_SPLITS}")
print(f"Run random forest           : {RUN_RANDOM_FOREST}")
print(f"Quick smoke test            : {QUICK_SMOKE_TEST}")
print(f"Model fitting authorized    : {MODEL_FITTING_AUTHORIZED}")
print(f"Validation after selection  : {OPEN_FROZEN_VALIDATION_AFTER_SELECTION}")
print(f"Hyperparameter tuning       : {HYPERPARAMETER_TUNING_ALLOWED}")
print(f"Clinical use allowed        : {CLINICAL_USE_ALLOWED}")
print("[PASS] Baseline governance initialized.")


Mounted at /content/drive
[CELL 2] GRABMyo BASELINE CONFIGURATION
NPZ path                    : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0/outputs/grabmyo-primary4-forearm16-fall14.npz
Runtime output              : /content/data/grabmyo-physionet-v1.1.0/baseline/grabmyo-primary4-baseline-v1
Persistent output           : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0/outputs/baseline/grabmyo-primary4-baseline-v1
Grouped CV folds            : 5
Run random forest           : False
Quick smoke test            : False
Model fitting authorized    : True
Validation after selection  : True
Hyperparameter tuning       : False
Clinical use allowed        : False
[PASS] Baseline governance initialized.


## CELL 3 — Common helpers và evaluation primitives

**Input:** arrays và estimators.  
**Output:** atomic artifact writing, score alignment, trial aggregation và metrics.


In [3]:
# === CELL 3: Common helpers and evaluation primitives ===
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_obj:
        while True:
            chunk = file_obj.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, np.generic):
        return json_safe(value.item())
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def write_json_atomic(payload: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".part")
    temporary.write_text(
        json.dumps(json_safe(payload), ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def write_dataframe_atomic(dataframe: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".part")
    compression = "gzip" if path.name.endswith(".csv.gz") else None
    dataframe.to_csv(temporary, index=False, compression=compression)
    temporary.replace(path)


def copy_atomic(source: Path, destination: Path) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    temporary.unlink(missing_ok=True)
    shutil.copy2(source, temporary)
    if temporary.stat().st_size != source.stat().st_size:
        temporary.unlink(missing_ok=True)
        raise RuntimeError(f"Copy size mismatch: {source} -> {destination}")
    temporary.replace(destination)
    return destination


def persist_artifact(path: Path) -> Path:
    return copy_atomic(path, PERSIST_ROOT / path.name)


def scalar_text(npz: Any, key: str, default: str = "") -> str:
    if key not in npz.files:
        return default
    value = np.asarray(npz[key])
    if value.size != 1:
        raise RuntimeError(f"Expected scalar key: {key}")
    return str(value.reshape(-1)[0])


def classification_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "cohen_kappa": float(cohen_kappa_score(y_true, y_pred)),
    }


def estimator_classes(estimator: Any) -> np.ndarray:
    if hasattr(estimator, "classes_"):
        return np.asarray(estimator.classes_).astype(str)
    if isinstance(estimator, Pipeline) and hasattr(estimator[-1], "classes_"):
        return np.asarray(estimator[-1].classes_).astype(str)
    raise RuntimeError("Estimator does not expose classes_.")


def aligned_score_matrix(
    estimator: Any,
    X_input: np.ndarray,
    class_order: np.ndarray,
) -> np.ndarray:
    if hasattr(estimator, "predict_proba"):
        raw = np.asarray(estimator.predict_proba(X_input), dtype=np.float64)
    elif hasattr(estimator, "decision_function"):
        raw = np.asarray(estimator.decision_function(X_input), dtype=np.float64)
        if raw.ndim == 1:
            raw = np.column_stack([-raw, raw])
    else:
        predictions = np.asarray(estimator.predict(X_input)).astype(str)
        raw = np.zeros((len(predictions), len(class_order)), dtype=np.float64)
        for class_index, label in enumerate(class_order):
            raw[:, class_index] = predictions == label
        return raw

    classes = estimator_classes(estimator)
    aligned = np.full((raw.shape[0], len(class_order)), -np.inf, dtype=np.float64)
    for source_index, label in enumerate(classes):
        matches = np.flatnonzero(class_order == label)
        if len(matches) != 1:
            raise RuntimeError(f"Estimator class not in frozen class order: {label}")
        aligned[:, int(matches[0])] = raw[:, source_index]
    if not np.isfinite(aligned).all():
        raise RuntimeError("Could not align all score columns to class_order.")
    return aligned


def aggregate_window_scores_to_trials(
    frame: pd.DataFrame,
    scores: np.ndarray,
    class_order: np.ndarray,
) -> pd.DataFrame:
    score_columns = [f"score__{label}" for label in class_order]
    score_frame = pd.DataFrame(scores, columns=score_columns)
    working = pd.concat([frame.reset_index(drop=True), score_frame], axis=1)

    group_columns = [
        "record_id", "repetition_id", "subject_id", "session_id",
        "gesture_id", "trial_id", "split_name", "y_true",
    ]
    grouped = working.groupby(group_columns, as_index=False, sort=False)
    aggregated = grouped[score_columns].mean()
    counts = grouped.size().rename(columns={"size": "valid_window_count"})
    result = aggregated.merge(counts, on=group_columns, how="left", validate="one_to_one")
    score_values = result[score_columns].to_numpy(dtype=np.float64)
    result["y_pred"] = class_order[np.argmax(score_values, axis=1)]
    result["expected_window_count"] = EXPECTED_WINDOWS_PER_RECORD
    result["valid_window_fraction"] = result["valid_window_count"] / EXPECTED_WINDOWS_PER_RECORD
    result["low_coverage_flag"] = result["valid_window_fraction"] < 0.80
    return result


def per_group_metrics(frame: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    rows = []
    for group_key, group in frame.groupby(group_columns, sort=True):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        row = {column: value for column, value in zip(group_columns, group_key)}
        row.update(classification_metrics(group["y_true"].to_numpy(), group["y_pred"].to_numpy()))
        row["n_samples"] = len(group)
        rows.append(row)
    return pd.DataFrame(rows)


def plot_confusion(cm: np.ndarray, labels: np.ndarray, title: str, path: Path) -> None:
    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(cm)
    fig.colorbar(image, ax=ax)
    ax.set_xticks(range(len(labels)), labels=labels, rotation=45, ha="right")
    ax.set_yticks(range(len(labels)), labels=labels)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(int(cm[i, j])), ha="center", va="center")
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)


print("[PASS] Modeling helpers initialized.")


[PASS] Modeling helpers initialized.


## CELL 4 — Locate, load và validate GRABMyo ETL artifact

**Input:** canonical NPZ và ETL manifests.  
**Output:** train/validation arrays không leakage; exclusion population được giải thích.


In [4]:
# === CELL 4: Load and validate ETL artifact ===
for required_path in [NPZ_PATH, RECORD_COVERAGE_PATH, EXCLUSION_REPORT_PATH, ETL_FINAL_GATE_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required ETL artifact not found: {required_path}")

with np.load(NPZ_PATH, allow_pickle=False) as data:
    required_keys = {
        "X", "y", "groups", "subject_id", "session_id", "gesture_id",
        "trial_id", "record_id", "repetition_id", "window_id",
        "window_ordinal", "window_start", "window_end", "split_names",
        "feature_names", "channel_names", "class_order",
        "F_TD8_indices", "F_SP6_indices", "F_ALL14_indices",
        "F_NO_MOMENTS12_indices", "dataset_view_id", "feature_set_version",
        "etl_contract_hash",
    }
    missing = required_keys - set(data.files)
    if missing:
        raise RuntimeError(f"NPZ missing keys: {sorted(missing)}")

    X = np.asarray(data["X"], dtype=np.float32)
    y = np.asarray(data["y"]).astype(str)
    groups = np.asarray(data["groups"], dtype=np.int16)
    subject_ids = np.asarray(data["subject_id"], dtype=np.int16)
    session_ids = np.asarray(data["session_id"], dtype=np.int8)
    gesture_ids = np.asarray(data["gesture_id"], dtype=np.int8)
    trial_ids = np.asarray(data["trial_id"], dtype=np.int8)
    record_ids = np.asarray(data["record_id"]).astype(str)
    repetition_ids = np.asarray(data["repetition_id"]).astype(str)
    window_ids = np.asarray(data["window_id"]).astype(str)
    window_ordinals = np.asarray(data["window_ordinal"], dtype=np.int16)
    window_starts = np.asarray(data["window_start"], dtype=np.int32)
    window_ends = np.asarray(data["window_end"], dtype=np.int32)
    split_names = np.asarray(data["split_names"]).astype(str)
    feature_names = np.asarray(data["feature_names"]).astype(str)
    channel_names = np.asarray(data["channel_names"]).astype(str)
    class_order = np.asarray(data["class_order"]).astype(str)
    feature_arms = {
        "F-TD8": np.asarray(data["F_TD8_indices"], dtype=np.int32),
        "F-SP6": np.asarray(data["F_SP6_indices"], dtype=np.int32),
        "F-ALL14": np.asarray(data["F_ALL14_indices"], dtype=np.int32),
        "F-NO-MOMENTS12": np.asarray(data["F_NO_MOMENTS12_indices"], dtype=np.int32),
    }
    dataset_view_id = scalar_text(data, "dataset_view_id")
    feature_set_version = scalar_text(data, "feature_set_version")
    etl_contract_hash = scalar_text(data, "etl_contract_hash")

if dataset_view_id != EXPECTED_DATASET_VIEW_ID:
    raise RuntimeError(f"Dataset view mismatch: {dataset_view_id}")
if tuple(class_order.tolist()) != EXPECTED_CLASS_ORDER:
    raise RuntimeError(f"Class order mismatch: {class_order.tolist()}")
if X.ndim != 2 or X.shape[1] != EXPECTED_FEATURE_DIM:
    raise RuntimeError(f"Unexpected X shape: {X.shape}")
if not np.isfinite(X).all():
    raise RuntimeError("X contains NaN/Inf.")

n_rows = X.shape[0]
for name, array in [
    ("y", y), ("groups", groups), ("subject_ids", subject_ids),
    ("session_ids", session_ids), ("gesture_ids", gesture_ids),
    ("trial_ids", trial_ids), ("record_ids", record_ids),
    ("repetition_ids", repetition_ids), ("window_ids", window_ids),
    ("split_names", split_names),
]:
    if len(array) != n_rows:
        raise RuntimeError(f"Row alignment failed: {name}")

if not np.array_equal(groups, subject_ids):
    raise RuntimeError("groups must equal subject_id for subject-independent evaluation.")
if len(window_ids) != len(np.unique(window_ids)):
    raise RuntimeError("Duplicate window IDs.")
if set(np.unique(y)) != set(class_order):
    raise RuntimeError("Observed classes do not match class_order.")

train_mask = split_names == "train"
validation_mask = split_names == "validation"
if not train_mask.any() or not validation_mask.any():
    raise RuntimeError("Both train and validation partitions are required.")
if np.any(~(train_mask | validation_mask)):
    raise RuntimeError(f"Unexpected split values: {np.unique(split_names)}")

train_subject_set = set(subject_ids[train_mask].astype(int).tolist())
validation_subject_set = set(subject_ids[validation_mask].astype(int).tolist())
if not train_subject_set.isdisjoint(validation_subject_set):
    raise RuntimeError("Subject leakage across train/validation.")
if len(train_subject_set) != EXPECTED_TRAIN_SUBJECTS:
    raise RuntimeError(f"Expected {EXPECTED_TRAIN_SUBJECTS} train subjects, got {len(train_subject_set)}")
if len(validation_subject_set) != EXPECTED_VALIDATION_SUBJECTS:
    raise RuntimeError(f"Expected {EXPECTED_VALIDATION_SUBJECTS} validation subjects, got {len(validation_subject_set)}")
if len(train_subject_set | validation_subject_set) != EXPECTED_SUBJECT_COUNT:
    raise RuntimeError("Total subject count mismatch.")

coverage_df = pd.read_csv(RECORD_COVERAGE_PATH)
try:
    exclusion_df = pd.read_csv(EXCLUSION_REPORT_PATH)
except pd.errors.EmptyDataError:
    exclusion_df = pd.DataFrame()
etl_final_gate = json.loads(ETL_FINAL_GATE_PATH.read_text(encoding="utf-8"))

if not etl_final_gate.get("status", "").startswith("PASS"):
    raise RuntimeError(f"ETL final gate is not PASS: {etl_final_gate}")
if len(coverage_df) != 43 * 3 * 4 * 7:
    raise RuntimeError(f"Expected 3612 coverage records, got {len(coverage_df)}")

def parse_bool_series(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)
    normalized = series.astype(str).str.strip().str.lower()
    allowed = {"true", "false", "1", "0"}
    unexpected = set(normalized.unique()) - allowed
    if unexpected:
        raise RuntimeError(f"Cannot parse boolean values: {sorted(unexpected)}")
    return normalized.isin({"true", "1"})

coverage_df["fully_excluded"] = parse_bool_series(coverage_df["fully_excluded"])
coverage_df["partially_excluded"] = parse_bool_series(coverage_df["partially_excluded"])
fully_excluded_validation = coverage_df[
    coverage_df["fully_excluded"] & (coverage_df["partition"] == "validation")
]
partially_excluded_validation = coverage_df[
    coverage_df["partially_excluded"] & (coverage_df["partition"] == "validation")
]
if not fully_excluded_validation.empty:
    raise RuntimeError(
        "Frozen validation contains fully excluded records; evaluation denominator requires review:\n"
        + fully_excluded_validation.to_json(orient="records", indent=2)
    )

input_gate = {
    "schema_version": "grabmyo-baseline-input-gate.v1",
    "created_at_utc": utc_now_iso(),
    "status": "PASS" if exclusion_df.empty else "PASS_WITH_DOCUMENTED_EXCLUSIONS",
    "npz_path": str(NPZ_PATH),
    "npz_sha256": sha256_file(NPZ_PATH),
    "dataset_view_id": dataset_view_id,
    "feature_set_version": feature_set_version,
    "etl_contract_hash": etl_contract_hash,
    "X_shape": list(X.shape),
    "subject_counts": {
        "total": len(train_subject_set | validation_subject_set),
        "train": len(train_subject_set),
        "validation": len(validation_subject_set),
    },
    "window_counts": {
        "total": n_rows,
        "train": int(train_mask.sum()),
        "validation": int(validation_mask.sum()),
    },
    "record_coverage": {
        "records": len(coverage_df),
        "fully_excluded": int(coverage_df["fully_excluded"].sum()),
        "partially_excluded": int(coverage_df["partially_excluded"].sum()),
        "validation_fully_excluded": len(fully_excluded_validation),
        "validation_partially_excluded": len(partially_excluded_validation),
    },
    "subject_overlap": sorted(train_subject_set & validation_subject_set),
}
INPUT_GATE_JSON = RUNTIME_ROOT / "grabmyo-baseline-input-gate.json"
write_json_atomic(input_gate, INPUT_GATE_JSON)
persist_artifact(INPUT_GATE_JSON)

print("=" * 92)
print("[CELL 4 INPUT GATE]")
print("=" * 92)
print(f"X shape                       : {X.shape}")
print(f"Classes                       : {class_order.tolist()}")
print(f"Channels                      : {len(channel_names)}")
print(f"Train subjects                : {len(train_subject_set)}")
print(f"Validation subjects           : {len(validation_subject_set)}")
print(f"Train windows                 : {int(train_mask.sum()):,}")
print(f"Validation windows            : {int(validation_mask.sum()):,}")
print(f"Fully excluded records        : {int(coverage_df['fully_excluded'].sum())}")
print(f"Partially excluded records    : {int(coverage_df['partially_excluded'].sum())}")
print(f"Validation full exclusions    : {len(fully_excluded_validation)}")
print(f"Input status                  : {input_gate['status']}")
print("[PASS] Leakage-safe baseline input is valid.")


[CELL 4 INPUT GATE]
X shape                       : (173376, 224)
Classes                       : ['rest', 'hand_close', 'wrist_flexion', 'wrist_extension']
Channels                      : 16
Train subjects                : 34
Validation subjects           : 9
Train windows                 : 137,088
Validation windows            : 36,288
Fully excluded records        : 0
Partially excluded records    : 0
Validation full exclusions    : 0
Input status                  : PASS
[PASS] Leakage-safe baseline input is valid.


## CELL 5 — Model registry và feature arms

**Input:** frozen feature indices từ NPZ.  
**Output:** deterministic candidate registry; không tuning trên validation.


In [5]:
# === CELL 5: Feature arms and model registry ===
for arm_name, indices in feature_arms.items():
    if indices.ndim != 1 or len(indices) == 0:
        raise RuntimeError(f"Invalid feature arm: {arm_name}")
    if int(indices.min()) < 0 or int(indices.max()) >= X.shape[1]:
        raise RuntimeError(f"Feature arm out of range: {arm_name}")

expected_arm_dims = {
    "F-TD8": 128,
    "F-SP6": 96,
    "F-ALL14": 224,
    "F-NO-MOMENTS12": 192,
}
for arm_name, expected_dim in expected_arm_dims.items():
    if len(feature_arms[arm_name]) != expected_dim:
        raise RuntimeError(
            f"Feature arm dimension mismatch {arm_name}: "
            f"expected={expected_dim}, actual={len(feature_arms[arm_name])}"
        )

model_registry: dict[str, Any] = {
    "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
    "lda_shrinkage": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")),
    ]),
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            C=1.0,
            max_iter=500,
            solver="lbfgs",
            class_weight=None,
            random_state=RANDOM_SEED,
        )),
    ]),
    "linear_svm": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearSVC(
            C=1.0,
            class_weight=None,
            dual=False,
            random_state=RANDOM_SEED,
            max_iter=10000,
        )),
    ]),
}

if RUN_RANDOM_FOREST:
    model_registry["random_forest"] = RandomForestClassifier(
        n_estimators=300,
        max_features="sqrt",
        min_samples_leaf=1,
        class_weight=None,
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS,
    )

print("[FEATURE ARMS]")
for arm_name, indices in feature_arms.items():
    print(f"  - {arm_name:<18}: {len(indices)} dimensions")
print("[MODELS]")
for model_name in model_registry:
    print(f"  - {model_name}")
print("[PASS] Frozen candidate registry initialized.")


[FEATURE ARMS]
  - F-TD8             : 128 dimensions
  - F-SP6             : 96 dimensions
  - F-ALL14           : 224 dimensions
  - F-NO-MOMENTS12    : 192 dimensions
[MODELS]
  - dummy_most_frequent
  - lda_shrinkage
  - logistic_regression
  - linear_svm
[PASS] Frozen candidate registry initialized.


## CELL 6 — Build subject-grouped CV folds

**Input:** 34 training subjects.  
**Output:** 5 disjoint folds; mọi window/session/trial của một subject đi cùng nhau.


In [6]:
# === CELL 6: Subject-grouped CV fold contract ===
train_indices_global = np.flatnonzero(train_mask)
y_train_all = y[train_mask]
groups_train_all = subject_ids[train_mask]

if QUICK_SMOKE_TEST:
    smoke_subjects = sorted(np.unique(groups_train_all))[:10]
    smoke_mask_local = np.isin(groups_train_all, smoke_subjects)
    train_indices_for_cv = train_indices_global[smoke_mask_local]
else:
    train_indices_for_cv = train_indices_global

y_cv = y[train_indices_for_cv]
groups_cv = subject_ids[train_indices_for_cv]

splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_SEED,
)
cv_splits = list(splitter.split(np.zeros(len(y_cv)), y_cv, groups_cv))

fold_assignment_rows = []
for fold_index, (fit_local, holdout_local) in enumerate(cv_splits):
    fit_subjects = set(groups_cv[fit_local].astype(int).tolist())
    holdout_subjects = set(groups_cv[holdout_local].astype(int).tolist())
    if not fit_subjects.isdisjoint(holdout_subjects):
        raise RuntimeError(f"Subject leakage in fold {fold_index}")
    for subject_id_value in sorted(holdout_subjects):
        fold_assignment_rows.append({
            "subject_id": subject_id_value,
            "fold": fold_index,
            "role": "oof_holdout",
        })

fold_assignment_df = pd.DataFrame(fold_assignment_rows)
if fold_assignment_df["subject_id"].duplicated().any():
    raise RuntimeError("A training subject appears in multiple holdout folds.")
if set(fold_assignment_df["subject_id"].astype(int)) != set(np.unique(groups_cv).astype(int)):
    raise RuntimeError("Not every CV subject has exactly one OOF fold.")

CV_FOLD_ASSIGNMENT_CSV = RUNTIME_ROOT / "grabmyo-cv-subject-fold-assignment.csv"
CV_FOLD_CONTRACT_JSON = RUNTIME_ROOT / "grabmyo-cv-fold-contract.json"
write_dataframe_atomic(fold_assignment_df, CV_FOLD_ASSIGNMENT_CSV)
write_json_atomic({
    "schema_version": "grabmyo-subject-grouped-cv.v1",
    "created_at_utc": utc_now_iso(),
    "n_splits": N_SPLITS,
    "random_seed": RANDOM_SEED,
    "group_key": "subject_id",
    "cv_subject_count": int(len(np.unique(groups_cv))),
    "cv_window_count": int(len(y_cv)),
    "quick_smoke_test": QUICK_SMOKE_TEST,
    "folds": [
        {
            "fold": fold_index,
            "fit_subject_ids": sorted(set(groups_cv[fit_local].astype(int).tolist())),
            "holdout_subject_ids": sorted(set(groups_cv[holdout_local].astype(int).tolist())),
            "fit_windows": int(len(fit_local)),
            "holdout_windows": int(len(holdout_local)),
        }
        for fold_index, (fit_local, holdout_local) in enumerate(cv_splits)
    ],
}, CV_FOLD_CONTRACT_JSON)
persist_artifact(CV_FOLD_ASSIGNMENT_CSV)
persist_artifact(CV_FOLD_CONTRACT_JSON)

print(f"[PASS] {N_SPLITS} subject-grouped folds created for {len(np.unique(groups_cv))} subjects.")


[PASS] 5 subject-grouped folds created for 34 subjects.


## CELL 7 — Grouped-CV benchmark với checkpoint

**Input:** model × feature-arm candidates.  
**Output:** fold metrics và leaderboard; validation chưa được mở.


In [7]:
# === CELL 7: Grouped-CV benchmark ===
# Persist candidate checkpoints directly so CV can resume after Colab reconnects.
CV_RESULTS_DIR = PERSIST_ROOT / "cv-candidates"
CV_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

base_meta = pd.DataFrame({
    "global_index": train_indices_for_cv,
    "y_true": y[train_indices_for_cv],
    "subject_id": subject_ids[train_indices_for_cv],
    "session_id": session_ids[train_indices_for_cv],
    "gesture_id": gesture_ids[train_indices_for_cv],
    "trial_id": trial_ids[train_indices_for_cv],
    "record_id": record_ids[train_indices_for_cv],
    "repetition_id": repetition_ids[train_indices_for_cv],
    "window_id": window_ids[train_indices_for_cv],
    "split_name": split_names[train_indices_for_cv],
})

fold_metric_rows = []
candidate_summary_rows = []

for arm_name, arm_indices in feature_arms.items():
    for model_name, model_template in model_registry.items():
        candidate_id = f"{arm_name}__{model_name}"
        candidate_path = CV_RESULTS_DIR / f"{candidate_id}.json"

        if candidate_path.exists() and not FORCE_RECOMPUTE_CV:
            payload = json.loads(candidate_path.read_text(encoding="utf-8"))
            fold_metric_rows.extend(payload["fold_metrics"])
            candidate_summary_rows.append(payload["summary"])
            print(f"[SKIP] {candidate_id}")
            continue

        print(f"[RUN] {candidate_id}")
        candidate_started = time.perf_counter()
        oof_predictions = np.empty(len(y_cv), dtype="U32")
        oof_scores = np.full((len(y_cv), len(class_order)), np.nan, dtype=np.float64)
        fold_rows = []

        for fold_index, (fit_local, holdout_local) in enumerate(cv_splits):
            estimator = clone(model_template)
            X_fit = X[train_indices_for_cv[fit_local]][:, arm_indices]
            y_fit = y_cv[fit_local]
            X_holdout = X[train_indices_for_cv[holdout_local]][:, arm_indices]
            y_holdout = y_cv[holdout_local]

            fold_started = time.perf_counter()
            estimator.fit(X_fit, y_fit)
            predictions = np.asarray(estimator.predict(X_holdout)).astype(str)
            scores = aligned_score_matrix(estimator, X_holdout, class_order)
            oof_predictions[holdout_local] = predictions
            oof_scores[holdout_local] = scores

            holdout_frame = base_meta.iloc[holdout_local].copy().reset_index(drop=True)
            holdout_frame["y_pred"] = predictions
            trial_frame = aggregate_window_scores_to_trials(holdout_frame, scores, class_order)

            window_metrics = classification_metrics(y_holdout, predictions)
            trial_metrics = classification_metrics(
                trial_frame["y_true"].to_numpy(),
                trial_frame["y_pred"].to_numpy(),
            )
            row = {
                "candidate_id": candidate_id,
                "feature_arm": arm_name,
                "model_name": model_name,
                "fold": fold_index,
                "feature_dimension": len(arm_indices),
                "fit_windows": len(fit_local),
                "holdout_windows": len(holdout_local),
                "holdout_trials": len(trial_frame),
                "holdout_subjects": len(np.unique(groups_cv[holdout_local])),
                "fit_seconds": time.perf_counter() - fold_started,
                **{f"window_{k}": v for k, v in window_metrics.items()},
                **{f"trial_{k}": v for k, v in trial_metrics.items()},
            }
            fold_rows.append(row)
            print(
                f"    fold={fold_index} trial_macro_f1={trial_metrics['macro_f1']:.4f} "
                f"window_macro_f1={window_metrics['macro_f1']:.4f}"
            )

        if np.isnan(oof_scores).any():
            raise RuntimeError(f"Incomplete OOF scores for {candidate_id}")

        oof_frame = base_meta.copy()
        oof_frame["y_pred"] = oof_predictions
        oof_trial_frame = aggregate_window_scores_to_trials(oof_frame, oof_scores, class_order)
        oof_window_metrics = classification_metrics(y_cv, oof_predictions)
        oof_trial_metrics = classification_metrics(
            oof_trial_frame["y_true"].to_numpy(),
            oof_trial_frame["y_pred"].to_numpy(),
        )

        summary = {
            "candidate_id": candidate_id,
            "feature_arm": arm_name,
            "model_name": model_name,
            "feature_dimension": len(arm_indices),
            "oof_window_macro_f1": oof_window_metrics["macro_f1"],
            "oof_window_balanced_accuracy": oof_window_metrics["balanced_accuracy"],
            "oof_trial_macro_f1": oof_trial_metrics["macro_f1"],
            "oof_trial_balanced_accuracy": oof_trial_metrics["balanced_accuracy"],
            "oof_trial_accuracy": oof_trial_metrics["accuracy"],
            "mean_fold_trial_macro_f1": float(np.mean([row["trial_macro_f1"] for row in fold_rows])),
            "std_fold_trial_macro_f1": float(np.std([row["trial_macro_f1"] for row in fold_rows], ddof=0)),
            "elapsed_seconds": time.perf_counter() - candidate_started,
        }
        payload = {
            "schema_version": "grabmyo-cv-candidate.v1",
            "created_at_utc": utc_now_iso(),
            "summary": summary,
            "fold_metrics": fold_rows,
        }
        write_json_atomic(payload, candidate_path)
        fold_metric_rows.extend(fold_rows)
        candidate_summary_rows.append(summary)

fold_metrics_df = pd.DataFrame(fold_metric_rows)
leaderboard_df = pd.DataFrame(candidate_summary_rows).sort_values(
    ["oof_trial_macro_f1", "oof_trial_balanced_accuracy", "oof_window_macro_f1"],
    ascending=[False, False, False],
).reset_index(drop=True)
leaderboard_df.insert(0, "rank", np.arange(1, len(leaderboard_df) + 1))

CV_FOLD_METRICS_CSV = RUNTIME_ROOT / "grabmyo-cv-fold-metrics.csv"
CV_LEADERBOARD_CSV = RUNTIME_ROOT / "grabmyo-cv-leaderboard.csv"
CV_LEADERBOARD_PNG = RUNTIME_ROOT / "grabmyo-cv-leaderboard.png"
write_dataframe_atomic(fold_metrics_df, CV_FOLD_METRICS_CSV)
write_dataframe_atomic(leaderboard_df, CV_LEADERBOARD_CSV)

fig, ax = plt.subplots(figsize=(10, max(5, 0.45 * len(leaderboard_df))))
plot_df = leaderboard_df.sort_values("oof_trial_macro_f1", ascending=True)
ax.barh(plot_df["candidate_id"], plot_df["oof_trial_macro_f1"])
ax.set_xlabel("OOF trial-level macro-F1")
ax.set_title("GRABMyo grouped-CV leaderboard")
ax.set_xlim(0, 1)
fig.tight_layout()
fig.savefig(CV_LEADERBOARD_PNG, dpi=160, bbox_inches="tight")
plt.close(fig)

for artifact in [CV_FOLD_METRICS_CSV, CV_LEADERBOARD_CSV, CV_LEADERBOARD_PNG]:
    persist_artifact(artifact)

display(leaderboard_df)
print("[PASS] Grouped-CV benchmark completed; frozen validation remains unopened.")


[RUN] F-TD8__dummy_most_frequent
    fold=0 trial_macro_f1=0.1000 window_macro_f1=0.1000
    fold=1 trial_macro_f1=0.1000 window_macro_f1=0.1000
    fold=2 trial_macro_f1=0.1000 window_macro_f1=0.1000
    fold=3 trial_macro_f1=0.1000 window_macro_f1=0.1000
    fold=4 trial_macro_f1=0.1000 window_macro_f1=0.1000
[RUN] F-TD8__lda_shrinkage
    fold=0 trial_macro_f1=0.9242 window_macro_f1=0.8318
    fold=1 trial_macro_f1=0.8838 window_macro_f1=0.8295
    fold=2 trial_macro_f1=0.9682 window_macro_f1=0.8777
    fold=3 trial_macro_f1=0.9063 window_macro_f1=0.8311
    fold=4 trial_macro_f1=0.9509 window_macro_f1=0.8644
[RUN] F-TD8__logistic_regression
    fold=0 trial_macro_f1=0.9497 window_macro_f1=0.9066
    fold=1 trial_macro_f1=0.9591 window_macro_f1=0.9218
    fold=2 trial_macro_f1=0.9881 window_macro_f1=0.9449
    fold=3 trial_macro_f1=0.9644 window_macro_f1=0.9267
    fold=4 trial_macro_f1=0.9675 window_macro_f1=0.9379
[RUN] F-TD8__linear_svm
    fold=0 trial_macro_f1=0.9528 window_mac

,rank,candidate_id,feature_arm,model_name,feature_dimension,oof_window_macro_f1,oof_window_balanced_accuracy,oof_trial_macro_f1,oof_trial_balanced_accuracy,oof_trial_accuracy,mean_fold_trial_macro_f1,std_fold_trial_macro_f1,elapsed_seconds
0,1,F-ALL14__linear_svm,F-ALL14,linear_svm,224,0.947083,0.947129,0.982492,0.982493,0.982493,0.982964,0.010144,1806.238917
1,2,F-NO-MOMENTS12__logistic_regression,F-NO-MOMENTS12,logistic_regression,192,0.947714,0.947712,0.981086,0.981092,0.981092,0.981431,0.009722,430.537777
2,3,F-NO-MOMENTS12__linear_svm,F-NO-MOMENTS12,linear_svm,192,0.947020,0.947070,0.981083,0.981092,0.981092,0.981490,0.009792,1335.604704
3,4,F-ALL14__logistic_regression,F-ALL14,logistic_regression,224,0.946637,0.946618,0.979691,0.979692,0.979692,0.980084,0.008940,425.886155
4,5,F-SP6__logistic_regression,F-SP6,logistic_regression,96,0.931096,0.931037,0.971262,0.971289,0.971289,0.971422,0.006346,376.395607
5,6,F-SP6__linear_svm,F-SP6,linear_svm,96,0.924024,0.923961,0.969537,0.969538,0.969538,0.969863,0.011251,492.329157
6,7,F-TD8__linear_svm,F-TD8,linear_svm,128,0.927247,0.927667,0.967375,0.967437,0.967437,0.967901,0.009574,318.701496
7,8,F-TD8__logistic_regression,F-TD8,logistic_regression,128,0.927027,0.927193,0.964967,0.964986,0.964986,0.965763,0.012692,348.010581
8,9,F-ALL14__lda_shrinkage,F-ALL14,lda_shrinkage,224,0.918712,0.917732,0.960567,0.960434,0.960434,0.960981,0.014914,24.117426
9,10,F-NO-MOMENTS12__lda_shrinkage,F-NO-MOMENTS12,lda_shrinkage,192,0.911401,0.910160,0.953011,0.952731,0.952731,0.953287,0.013728,19.355191


[PASS] Grouped-CV benchmark completed; frozen validation remains unopened.


## CELL 8 — Select candidate và regenerate selected OOF evidence

**Input:** CV leaderboard only.  
**Output:** immutable selection record và selected-candidate OOF predictions.


In [8]:
# === CELL 8: Model selection and selected OOF evidence ===
if leaderboard_df.empty:
    raise RuntimeError("CV leaderboard is empty.")

selected_row = leaderboard_df.iloc[0]
selected_candidate_id = str(selected_row["candidate_id"])
selected_feature_arm = str(selected_row["feature_arm"])
selected_model_name = str(selected_row["model_name"])
selected_indices = feature_arms[selected_feature_arm]
selected_template = model_registry[selected_model_name]

selection_payload = {
    "schema_version": "grabmyo-model-selection.v1",
    "created_at_utc": utc_now_iso(),
    "selection_metric": "OOF trial-level macro-F1 from subject-grouped CV",
    "tie_breakers": ["OOF trial balanced accuracy", "OOF window macro-F1"],
    "selected_candidate_id": selected_candidate_id,
    "selected_feature_arm": selected_feature_arm,
    "selected_model_name": selected_model_name,
    "selected_feature_dimension": len(selected_indices),
    "selected_cv_metrics": selected_row.to_dict(),
    "validation_used_for_selection": False,
    "hyperparameter_tuning_allowed": False,
}
MODEL_SELECTION_JSON = RUNTIME_ROOT / "grabmyo-model-selection.json"
write_json_atomic(selection_payload, MODEL_SELECTION_JSON)
persist_artifact(MODEL_SELECTION_JSON)

# Regenerate exact selected-candidate OOF predictions for handoff.
oof_predictions = np.empty(len(y_cv), dtype="U32")
oof_scores = np.full((len(y_cv), len(class_order)), np.nan, dtype=np.float64)
oof_fold = np.full(len(y_cv), -1, dtype=np.int8)

for fold_index, (fit_local, holdout_local) in enumerate(cv_splits):
    estimator = clone(selected_template)
    estimator.fit(
        X[train_indices_for_cv[fit_local]][:, selected_indices],
        y_cv[fit_local],
    )
    X_holdout = X[train_indices_for_cv[holdout_local]][:, selected_indices]
    oof_predictions[holdout_local] = np.asarray(estimator.predict(X_holdout)).astype(str)
    oof_scores[holdout_local] = aligned_score_matrix(estimator, X_holdout, class_order)
    oof_fold[holdout_local] = fold_index

if np.any(oof_fold < 0) or np.isnan(oof_scores).any():
    raise RuntimeError("Selected OOF regeneration incomplete.")

oof_window_df = base_meta.copy()
oof_window_df["fold"] = oof_fold
oof_window_df["y_pred"] = oof_predictions
for class_index, label in enumerate(class_order):
    oof_window_df[f"score__{label}"] = oof_scores[:, class_index]

oof_trial_df = aggregate_window_scores_to_trials(base_meta, oof_scores, class_order)
oof_subject_metrics_df = per_group_metrics(oof_trial_df, ["subject_id"])
oof_session_metrics_df = per_group_metrics(oof_trial_df, ["session_id"])
oof_metrics = {
    "window": classification_metrics(y_cv, oof_predictions),
    "trial": classification_metrics(oof_trial_df["y_true"], oof_trial_df["y_pred"]),
    "low_coverage_trials": int(oof_trial_df["low_coverage_flag"].sum()),
}

OOF_WINDOW_CSV_GZ = RUNTIME_ROOT / "grabmyo-selected-oof-window-predictions.csv.gz"
OOF_TRIAL_CSV = RUNTIME_ROOT / "grabmyo-selected-oof-trial-predictions.csv"
OOF_SUBJECT_METRICS_CSV = RUNTIME_ROOT / "grabmyo-selected-oof-subject-metrics.csv"
OOF_SESSION_METRICS_CSV = RUNTIME_ROOT / "grabmyo-selected-oof-session-metrics.csv"
OOF_METRICS_JSON = RUNTIME_ROOT / "grabmyo-selected-oof-metrics.json"

write_dataframe_atomic(oof_window_df, OOF_WINDOW_CSV_GZ)
write_dataframe_atomic(oof_trial_df, OOF_TRIAL_CSV)
write_dataframe_atomic(oof_subject_metrics_df, OOF_SUBJECT_METRICS_CSV)
write_dataframe_atomic(oof_session_metrics_df, OOF_SESSION_METRICS_CSV)
write_json_atomic(oof_metrics, OOF_METRICS_JSON)
for artifact in [
    OOF_WINDOW_CSV_GZ, OOF_TRIAL_CSV, OOF_SUBJECT_METRICS_CSV,
    OOF_SESSION_METRICS_CSV, OOF_METRICS_JSON,
]:
    persist_artifact(artifact)

print(json.dumps(json_safe(selection_payload), indent=2))
print("[PASS] Candidate selected using grouped-CV only.")


{
  "schema_version": "grabmyo-model-selection.v1",
  "created_at_utc": "2026-08-03T08:35:10.006091+00:00",
  "selection_metric": "OOF trial-level macro-F1 from subject-grouped CV",
  "tie_breakers": [
    "OOF trial balanced accuracy",
    "OOF window macro-F1"
  ],
  "selected_candidate_id": "F-ALL14__linear_svm",
  "selected_feature_arm": "F-ALL14",
  "selected_model_name": "linear_svm",
  "selected_feature_dimension": 224,
  "selected_cv_metrics": {
    "rank": 1,
    "candidate_id": "F-ALL14__linear_svm",
    "feature_arm": "F-ALL14",
    "model_name": "linear_svm",
    "feature_dimension": 224,
    "oof_window_macro_f1": 0.9470826269493727,
    "oof_window_balanced_accuracy": 0.9471288515406163,
    "oof_trial_macro_f1": 0.9824917282762433,
    "oof_trial_balanced_accuracy": 0.9824929971988796,
    "oof_trial_accuracy": 0.9824929971988795,
    "mean_fold_trial_macro_f1": 0.9829642676947087,
    "std_fold_trial_macro_f1": 0.010144240856036364,
    "elapsed_seconds": 1806.238916873

## CELL 9 — Refit selected model và open frozen validation once

**Input:** selected candidate, all 34 train subjects.  
**Output:** frozen validation predictions và multi-level metrics.


In [9]:
# === CELL 9: Refit and single frozen-validation evaluation ===
if not OPEN_FROZEN_VALIDATION_AFTER_SELECTION:
    raise RuntimeError("Frozen validation opening is not authorized.")

selected_model = clone(selected_template)
fit_started = time.perf_counter()
selected_model.fit(X[train_mask][:, selected_indices], y[train_mask])
refit_seconds = time.perf_counter() - fit_started

X_validation = X[validation_mask][:, selected_indices]
y_validation = y[validation_mask]
validation_predictions = np.asarray(selected_model.predict(X_validation)).astype(str)
validation_scores = aligned_score_matrix(selected_model, X_validation, class_order)

validation_base = pd.DataFrame({
    "global_index": np.flatnonzero(validation_mask),
    "y_true": y_validation,
    "subject_id": subject_ids[validation_mask],
    "session_id": session_ids[validation_mask],
    "gesture_id": gesture_ids[validation_mask],
    "trial_id": trial_ids[validation_mask],
    "record_id": record_ids[validation_mask],
    "repetition_id": repetition_ids[validation_mask],
    "window_id": window_ids[validation_mask],
    "split_name": split_names[validation_mask],
})
validation_window_df = validation_base.copy()
validation_window_df["y_pred"] = validation_predictions
for class_index, label in enumerate(class_order):
    validation_window_df[f"score__{label}"] = validation_scores[:, class_index]

validation_trial_df = aggregate_window_scores_to_trials(
    validation_base, validation_scores, class_order
)
validation_subject_metrics_df = per_group_metrics(validation_trial_df, ["subject_id"])
validation_session_metrics_df = per_group_metrics(validation_trial_df, ["session_id"])
validation_subject_session_metrics_df = per_group_metrics(
    validation_trial_df, ["subject_id", "session_id"]
)

validation_metrics = {
    "schema_version": "grabmyo-validation-metrics.v1",
    "created_at_utc": utc_now_iso(),
    "selected_candidate_id": selected_candidate_id,
    "window": classification_metrics(y_validation, validation_predictions),
    "trial": classification_metrics(
        validation_trial_df["y_true"].to_numpy(),
        validation_trial_df["y_pred"].to_numpy(),
    ),
    "trial_classification_report": classification_report(
        validation_trial_df["y_true"],
        validation_trial_df["y_pred"],
        labels=class_order,
        output_dict=True,
        zero_division=0,
    ),
    "validation_subject_count": len(validation_subject_set),
    "validation_window_count": len(validation_window_df),
    "validation_trial_count": len(validation_trial_df),
    "low_coverage_trials": int(validation_trial_df["low_coverage_flag"].sum()),
    "refit_seconds": refit_seconds,
    "validation_open_count": 1,
}

window_cm = confusion_matrix(y_validation, validation_predictions, labels=class_order)
trial_cm = confusion_matrix(
    validation_trial_df["y_true"], validation_trial_df["y_pred"], labels=class_order
)

VALIDATION_WINDOW_CSV_GZ = RUNTIME_ROOT / "grabmyo-validation-window-predictions.csv.gz"
VALIDATION_TRIAL_CSV = RUNTIME_ROOT / "grabmyo-validation-trial-predictions.csv"
VALIDATION_SUBJECT_METRICS_CSV = RUNTIME_ROOT / "grabmyo-validation-subject-metrics.csv"
VALIDATION_SESSION_METRICS_CSV = RUNTIME_ROOT / "grabmyo-validation-session-metrics.csv"
VALIDATION_SUBJECT_SESSION_METRICS_CSV = RUNTIME_ROOT / "grabmyo-validation-subject-session-metrics.csv"
VALIDATION_METRICS_JSON = RUNTIME_ROOT / "grabmyo-validation-metrics.json"
VALIDATION_WINDOW_CM_CSV = RUNTIME_ROOT / "grabmyo-validation-window-confusion-matrix.csv"
VALIDATION_TRIAL_CM_CSV = RUNTIME_ROOT / "grabmyo-validation-trial-confusion-matrix.csv"
VALIDATION_WINDOW_CM_PNG = RUNTIME_ROOT / "grabmyo-validation-window-confusion-matrix.png"
VALIDATION_TRIAL_CM_PNG = RUNTIME_ROOT / "grabmyo-validation-trial-confusion-matrix.png"

write_dataframe_atomic(validation_window_df, VALIDATION_WINDOW_CSV_GZ)
write_dataframe_atomic(validation_trial_df, VALIDATION_TRIAL_CSV)
write_dataframe_atomic(validation_subject_metrics_df, VALIDATION_SUBJECT_METRICS_CSV)
write_dataframe_atomic(validation_session_metrics_df, VALIDATION_SESSION_METRICS_CSV)
write_dataframe_atomic(validation_subject_session_metrics_df, VALIDATION_SUBJECT_SESSION_METRICS_CSV)
write_json_atomic(validation_metrics, VALIDATION_METRICS_JSON)
write_dataframe_atomic(pd.DataFrame(window_cm, index=class_order, columns=class_order).reset_index(names="true_label"), VALIDATION_WINDOW_CM_CSV)
write_dataframe_atomic(pd.DataFrame(trial_cm, index=class_order, columns=class_order).reset_index(names="true_label"), VALIDATION_TRIAL_CM_CSV)
plot_confusion(window_cm, class_order, "GRABMyo validation — window level", VALIDATION_WINDOW_CM_PNG)
plot_confusion(trial_cm, class_order, "GRABMyo validation — trial level", VALIDATION_TRIAL_CM_PNG)

for artifact in [
    VALIDATION_WINDOW_CSV_GZ, VALIDATION_TRIAL_CSV,
    VALIDATION_SUBJECT_METRICS_CSV, VALIDATION_SESSION_METRICS_CSV,
    VALIDATION_SUBJECT_SESSION_METRICS_CSV, VALIDATION_METRICS_JSON,
    VALIDATION_WINDOW_CM_CSV, VALIDATION_TRIAL_CM_CSV,
    VALIDATION_WINDOW_CM_PNG, VALIDATION_TRIAL_CM_PNG,
]:
    persist_artifact(artifact)

print(json.dumps(json_safe(validation_metrics), indent=2))
print("[PASS] Frozen validation opened once after model selection.")


{
  "schema_version": "grabmyo-validation-metrics.v1",
  "created_at_utc": "2026-08-03T09:12:43.705992+00:00",
  "selected_candidate_id": "F-ALL14__linear_svm",
  "window": {
    "accuracy": 0.966655643738977,
    "balanced_accuracy": 0.966655643738977,
    "macro_f1": 0.9666477883204774,
    "weighted_f1": 0.9666477883204774,
    "macro_precision": 0.9669190658703838,
    "macro_recall": 0.966655643738977,
    "cohen_kappa": 0.9555408583186361
  },
  "trial": {
    "accuracy": 0.9880952380952381,
    "balanced_accuracy": 0.9880952380952381,
    "macro_f1": 0.9880755734928758,
    "weighted_f1": 0.9880755734928759,
    "macro_precision": 0.9882703420711658,
    "macro_recall": 0.9880952380952381,
    "cohen_kappa": 0.9841269841269842
  },
  "trial_classification_report": {
    "rest": {
      "precision": 0.9894736842105263,
      "recall": 0.9947089947089947,
      "f1-score": 0.9920844327176781,
      "support": 189.0
    },
    "hand_close": {
      "precision": 0.994535519125683,
 

## CELL 10 — Save model, evidence, final gate và handoff ZIP

**Input:** selected fitted model và all evidence.  
**Output:** complete baseline handoff package cho error analysis/model card.


In [10]:
# === CELL 10: Model artifact, evidence, final gate and handoff ===
MODEL_JOBLIB = RUNTIME_ROOT / "grabmyo-selected-model.joblib"
MODEL_METADATA_JSON = RUNTIME_ROOT / "grabmyo-selected-model-metadata.json"
BASELINE_EVIDENCE_JSON = RUNTIME_ROOT / "grabmyo-baseline-evidence.json"
FINAL_GATE_JSON = RUNTIME_ROOT / "grabmyo-baseline-final-gate.json"
HANDOFF_ZIP = RUNTIME_ROOT / "grabmyo-baseline-handoff.zip"

joblib.dump(selected_model, MODEL_JOBLIB, compress=3)
model_metadata = {
    "schema_version": "grabmyo-selected-model-metadata.v1",
    "created_at_utc": utc_now_iso(),
    "run_id": RUN_ID,
    "dataset_id": DATASET_ID,
    "dataset_view_id": dataset_view_id,
    "npz_sha256": sha256_file(NPZ_PATH),
    "selected_candidate_id": selected_candidate_id,
    "feature_arm": selected_feature_arm,
    "feature_indices": selected_indices.tolist(),
    "feature_names": feature_names[selected_indices].tolist(),
    "class_order": class_order.tolist(),
    "train_subject_ids": sorted(train_subject_set),
    "validation_subject_ids": sorted(validation_subject_set),
    "random_seed": RANDOM_SEED,
    "sklearn_version": sklearn.__version__,
    "python_version": platform.python_version(),
    "clinical_use_allowed": False,
}
write_json_atomic(model_metadata, MODEL_METADATA_JSON)

baseline_evidence = {
    "schema_version": "grabmyo-baseline-evidence.v1",
    "created_at_utc": utc_now_iso(),
    "run_id": RUN_ID,
    "input_gate": input_gate,
    "cv": {
        "n_splits": N_SPLITS,
        "group_key": "subject_id",
        "candidate_count": len(leaderboard_df),
        "selection": selection_payload,
    },
    "validation": validation_metrics,
    "governance": {
        "model_fitting_authorized": MODEL_FITTING_AUTHORIZED,
        "validation_open_count": 1,
        "hyperparameter_tuning_allowed": HYPERPARAMETER_TUNING_ALLOWED,
        "test_set_opened": TEST_SET_OPENED,
        "pooled_training_allowed": POOLED_TRAINING_ALLOWED,
        "clinical_use_allowed": CLINICAL_USE_ALLOWED,
    },
    "artifacts": {
        "model_joblib": str(MODEL_JOBLIB),
        "model_sha256": sha256_file(MODEL_JOBLIB),
        "cv_leaderboard": str(CV_LEADERBOARD_CSV),
        "validation_metrics": str(VALIDATION_METRICS_JSON),
        "validation_trial_predictions": str(VALIDATION_TRIAL_CSV),
    },
}
write_json_atomic(baseline_evidence, BASELINE_EVIDENCE_JSON)

final_gate = {
    "schema_version": "grabmyo-baseline-final-gate.v1",
    "created_at_utc": utc_now_iso(),
    "status": "PASS",
    "checks": {
        "input_gate_pass": input_gate["status"].startswith("PASS"),
        "subject_split_disjoint": train_subject_set.isdisjoint(validation_subject_set),
        "cv_group_key_subject": True,
        "selected_without_validation": selection_payload["validation_used_for_selection"] is False,
        "validation_open_count_one": validation_metrics["validation_open_count"] == 1,
        "model_artifact_exists": MODEL_JOBLIB.exists(),
        "finite_validation_scores": bool(np.isfinite(validation_scores).all()),
        "test_set_opened": TEST_SET_OPENED,
        "pooled_training_allowed": POOLED_TRAINING_ALLOWED,
        "clinical_use_allowed": CLINICAL_USE_ALLOWED,
    },
    "selected_candidate_id": selected_candidate_id,
    "validation_trial_macro_f1": validation_metrics["trial"]["macro_f1"],
    "validation_window_macro_f1": validation_metrics["window"]["macro_f1"],
}

required_true = [
    "input_gate_pass", "subject_split_disjoint", "cv_group_key_subject",
    "selected_without_validation", "validation_open_count_one",
    "model_artifact_exists", "finite_validation_scores",
]
if not all(final_gate["checks"][name] for name in required_true):
    final_gate["status"] = "FAIL"
    write_json_atomic(final_gate, FINAL_GATE_JSON)
    raise RuntimeError(json.dumps(final_gate, indent=2))
if final_gate["checks"]["test_set_opened"] is not False:
    raise RuntimeError("Test set must remain closed.")
if final_gate["checks"]["pooled_training_allowed"] is not False:
    raise RuntimeError("Pooled training is forbidden.")
if final_gate["checks"]["clinical_use_allowed"] is not False:
    raise RuntimeError("Clinical use must remain forbidden.")

write_json_atomic(final_gate, FINAL_GATE_JSON)

handoff_artifacts = [
    INPUT_GATE_JSON,
    CV_FOLD_ASSIGNMENT_CSV,
    CV_FOLD_CONTRACT_JSON,
    CV_FOLD_METRICS_CSV,
    CV_LEADERBOARD_CSV,
    CV_LEADERBOARD_PNG,
    MODEL_SELECTION_JSON,
    OOF_TRIAL_CSV,
    OOF_SUBJECT_METRICS_CSV,
    OOF_SESSION_METRICS_CSV,
    OOF_METRICS_JSON,
    VALIDATION_TRIAL_CSV,
    VALIDATION_SUBJECT_METRICS_CSV,
    VALIDATION_SESSION_METRICS_CSV,
    VALIDATION_SUBJECT_SESSION_METRICS_CSV,
    VALIDATION_METRICS_JSON,
    VALIDATION_WINDOW_CM_CSV,
    VALIDATION_TRIAL_CM_CSV,
    VALIDATION_WINDOW_CM_PNG,
    VALIDATION_TRIAL_CM_PNG,
    MODEL_JOBLIB,
    MODEL_METADATA_JSON,
    BASELINE_EVIDENCE_JSON,
    FINAL_GATE_JSON,
]

with zipfile.ZipFile(HANDOFF_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for artifact in handoff_artifacts:
        if not artifact.exists():
            raise FileNotFoundError(f"Missing handoff artifact: {artifact}")
        archive.write(artifact, arcname=artifact.name)

for artifact in [MODEL_JOBLIB, MODEL_METADATA_JSON, BASELINE_EVIDENCE_JSON, FINAL_GATE_JSON, HANDOFF_ZIP]:
    persist_artifact(artifact)

persistent_zip = PERSIST_ROOT / HANDOFF_ZIP.name
print("=" * 92)
print("[GRABMyo BASELINE FINAL GATE]")
print("=" * 92)
print(f"Selected candidate             : {selected_candidate_id}")
print(f"Validation trial macro-F1      : {validation_metrics['trial']['macro_f1']:.6f}")
print(f"Validation window macro-F1     : {validation_metrics['window']['macro_f1']:.6f}")
print(f"Model artifact                 : {PERSIST_ROOT / MODEL_JOBLIB.name}")
print(f"Handoff ZIP                    : {persistent_zip}")
print(f"Handoff ZIP SHA-256            : {sha256_file(persistent_zip)}")
print(f"Test set opened                : {TEST_SET_OPENED}")
print(f"Clinical use allowed           : {CLINICAL_USE_ALLOWED}")
print("[PASS] GRABMyo baseline modeling completed.")

if IN_COLAB and DOWNLOAD_HANDOFF_TO_BROWSER:
    files.download(str(persistent_zip))


[GRABMyo BASELINE FINAL GATE]
Selected candidate             : F-ALL14__linear_svm
Validation trial macro-F1      : 0.988076
Validation window macro-F1     : 0.966648
Model artifact                 : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0/outputs/baseline/grabmyo-primary4-baseline-v1/grabmyo-selected-model.joblib
Handoff ZIP                    : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0/outputs/baseline/grabmyo-primary4-baseline-v1/grabmyo-baseline-handoff.zip
Handoff ZIP SHA-256            : 1592bd7738a079ac388dcff48392f5e3b2c52b40a21634313be94381af0680a1
Test set opened                : False
Clinical use allowed           : False
[PASS] GRABMyo baseline modeling completed.


# Main deliverable after PASS

Download hoặc copy file:

```text
/content/drive/MyDrive/MyoLab-AI-data/
└── grabmyo-physionet-v1.1.0/
    └── outputs/
        └── baseline/
            └── grabmyo-primary4-baseline-v1/
                └── grabmyo-baseline-handoff.zip
```

Window-level prediction CSV được giữ ngoài handoff ZIP để tránh package quá lớn; trial/subject/session artifacts và model đã có trong handoff.
